In [1]:
import pandas as pd
import os

In [2]:
import pandas as pd
import os

# Original cleaned dataset
input_file = r"C:\Users\prave\OneDrive\Desktop\PAYMENT FRAUD ANALYTICS\CLEAN DATA"

# Output folder
output_folder = r"C:\Users\prave\OneDrive\Desktop\PAYMENT FRAUD ANALYTICS\DATA PREPARATION"

os.makedirs(output_folder, exist_ok=True)

output_file = os.path.join(
    output_folder,
    "fraud_summary.csv"
)

# Required columns
usecols = [
    "step",
    "type",
    "amount",
    "isFraud",
    "isFlaggedFraud",
    "fraud_status",
    "flag_status",
    "amount_bucket",
    "orgin_balance_change",
    "destination_balance_change"
]

# Store smaller summaries
summary_list = []

# Process 1 lakh rows at a time
for chunk in pd.read_csv(
    input_file,
    usecols=usecols,
    chunksize=100000
):

    # Create fraud amount
    chunk["fraud_amount"] = (
        chunk["amount"] * chunk["isFraud"]
    )

    # Group and summarize
    summary = chunk.groupby(
        ["step", "type", "amount_bucket",
         "fraud_status", "flag_status"]
    ).agg(
        total_transactions=("amount", "size"),
        total_amount=("amount", "sum"),
        average_amount=("amount", "mean"),
        fraud_transactions=("isFraud", "sum"),
        flagged_transactions=("isFlaggedFraud", "sum"),
        fraud_amount=("fraud_amount", "sum"),
        sender_balance_change=("orgin_balance_change", "sum"),
        destination_balance_change=("destination_balance_change", "sum")
    ).reset_index()

    summary_list.append(summary)

# Combine summaries
final_summary = pd.concat(
    summary_list,
    ignore_index=True
)

# Final aggregation
final_summary = final_summary.groupby(
    ["step", "type", "amount_bucket",
     "fraud_status", "flag_status"],
    as_index=False
).agg(
    total_transactions=("total_transactions", "sum"),
    total_amount=("total_amount", "sum"),
    fraud_transactions=("fraud_transactions", "sum"),
    flagged_transactions=("flagged_transactions", "sum"),
    fraud_amount=("fraud_amount", "sum"),
    sender_balance_change=("sender_balance_change", "sum"),
    destination_balance_change=("destination_balance_change", "sum")
)

final_summary["average_amount"] = (
    final_summary["total_amount"] /
    final_summary["total_transactions"]
)

# Export CSV
final_summary.to_csv(output_file, index=False)

print("Power BI data preparation completed!")
print("Summary records:", len(final_summary))
print("Output file:", output_file)

PermissionError: [Errno 13] Permission denied: 'C:\\Users\\prave\\OneDrive\\Desktop\\PAYMENT FRAUD ANALYTICS\\CLEAN DATA'

In [3]:

import pandas as pd
import os
import gc

# =====================================
# 1. FILE PATHS
# =====================================

input_file = r"C:\Users\prave\OneDrive\Desktop\PAYMENT FRAUD ANALYTICS\CLEAN DATA\cleaned_data.csv"

output_folder = r"C:\Users\prave\OneDrive\Desktop\PAYMENT FRAUD ANALYTICS\DATA PREPARATION\OUTPUT FILE"

os.makedirs(output_folder, exist_ok=True)

output_file = os.path.join(
    output_folder,
    "fraud_summary.csv"
)

# =====================================
# 2. VERIFY FILE
# =====================================

if not os.path.isfile(input_file):
    raise FileNotFoundError(input_file)

print("Input file verified!")
print("Starting data preparation...")

# =====================================
# 3. REQUIRED COLUMNS
# =====================================

usecols = [
    "step",
    "type",
    "amount",
    "isFraud",
    "isFlaggedFraud",
    "orgin_balance_change",
    "destination_balance_change",
    "fraud_status",
    "flag_status",
    "amount_bucket"
]

# =====================================
# 4. CHUNK PROCESSING
# =====================================

summary_list = []

chunksize = 100000

for chunk_no, chunk in enumerate(
    pd.read_csv(
        input_file,
        usecols=usecols,
        chunksize=chunksize
    ),
    start=1
):

    print(f"Processing chunk: {chunk_no}")

    numeric_columns = [
        "step",
        "amount",
        "isFraud",
        "isFlaggedFraud",
        "orgin_balance_change",
        "destination_balance_change"
    ]

    for col in numeric_columns:
        chunk[col] = pd.to_numeric(
            chunk[col],
            errors="coerce"
        )

    # Calculate fraud amount
    chunk["fraud_amount"] = (
        chunk["amount"] * chunk["isFraud"]
    )

    # Aggregate transactions
    summary = chunk.groupby(
        [
            "step",
            "type",
            "amount_bucket",
            "fraud_status",
            "flag_status"
        ],
        dropna=False,
        observed=True
    ).agg(
        total_transactions=("amount", "size"),
        total_amount=("amount", "sum"),
        fraud_transactions=("isFraud", "sum"),
        flagged_transactions=("isFlaggedFraud", "sum"),
        fraud_amount=("fraud_amount", "sum"),
        sender_balance_change=("orgin_balance_change", "sum"),
        destination_balance_change=("destination_balance_change", "sum")
    ).reset_index()

    summary_list.append(summary)

    del chunk, summary
    gc.collect()

# =====================================
# 5. COMBINE SUMMARIES
# =====================================

print("Combining summary data...")

final_summary = pd.concat(
    summary_list,
    ignore_index=True
)

final_summary = final_summary.groupby(
    [
        "step",
        "type",
        "amount_bucket",
        "fraud_status",
        "flag_status"
    ],
    dropna=False,
    observed=True
).agg(
    total_transactions=("total_transactions", "sum"),
    total_amount=("total_amount", "sum"),
    fraud_transactions=("fraud_transactions", "sum"),
    flagged_transactions=("flagged_transactions", "sum"),
    fraud_amount=("fraud_amount", "sum"),
    sender_balance_change=("sender_balance_change", "sum"),
    destination_balance_change=("destination_balance_change", "sum")
).reset_index()

# =====================================
# 6. CALCULATE KPIs
# =====================================

final_summary["average_amount"] = (
    final_summary["total_amount"] /
    final_summary["total_transactions"]
)

final_summary["fraud_rate"] = (
    final_summary["fraud_transactions"] /
    final_summary["total_transactions"]
)

# =====================================
# 7. EXPORT CSV
# =====================================

final_summary.to_csv(
    output_file,
    index=False
)

# =====================================
# 8. VALIDATE OUTPUT
# =====================================

print("\n========== COMPLETED ==========")

print("Total transactions:",
      final_summary["total_transactions"].sum())

print("Fraud transactions:",
      final_summary["fraud_transactions"].sum())

print("Fraud amount:",
      final_summary["fraud_amount"].sum())

print("Summary rows:",
      len(final_summary))

print("Saved location:")
print(output_file)

print("================================")


Input file verified!
Starting data preparation...
Processing chunk: 1
Processing chunk: 2
Processing chunk: 3
Processing chunk: 4
Processing chunk: 5
Processing chunk: 6
Processing chunk: 7
Processing chunk: 8
Processing chunk: 9
Processing chunk: 10
Processing chunk: 11
Processing chunk: 12
Processing chunk: 13
Processing chunk: 14
Processing chunk: 15
Processing chunk: 16
Processing chunk: 17
Processing chunk: 18
Processing chunk: 19
Processing chunk: 20
Processing chunk: 21
Processing chunk: 22
Processing chunk: 23
Processing chunk: 24
Processing chunk: 25
Processing chunk: 26
Processing chunk: 27
Processing chunk: 28
Processing chunk: 29
Processing chunk: 30
Processing chunk: 31
Processing chunk: 32
Processing chunk: 33
Processing chunk: 34
Processing chunk: 35
Processing chunk: 36
Processing chunk: 37
Processing chunk: 38
Processing chunk: 39
Processing chunk: 40
Processing chunk: 41
Processing chunk: 42
Processing chunk: 43
Processing chunk: 44
Processing chunk: 45
Processing chu

In [4]:
import pandas as pd
import os

# Input file
input_file = r"C:\Users\prave\OneDrive\Desktop\PAYMENT FRAUD ANALYTICS\CLEAN DATA\cleaned_data.csv"

# Output file
output_file = r"C:\Users\prave\OneDrive\Desktop\PAYMENT FRAUD ANALYTICS\DATA PREPARATION\OUTPUT FILE\fraud_transactions.csv"

# Remove old output file if it exists
if os.path.exists(output_file):
    os.remove(output_file)

chunksize = 100000

total_rows = 0
total_fraud = 0
first_chunk = True

# Read data in chunks
for chunk in pd.read_csv(input_file, chunksize=chunksize):

    total_rows += len(chunk)

    # Convert fraud column to numeric
    chunk["isFraud"] = pd.to_numeric(
        chunk["isFraud"], errors="coerce"
    )

    # Extract fraud transactions
    fraud_data = chunk[chunk["isFraud"] == 1]

    if not fraud_data.empty:

        fraud_data.to_csv(
            output_file,
            mode="a",
            index=False,
            header=first_chunk
        )

        total_fraud += len(fraud_data)
        first_chunk = False

    print(f"Processed: {total_rows:,} | Fraud: {total_fraud:,}")

print("\nPROCESS COMPLETED!")
print("Total Records:", total_rows)
print("Total Fraud Transactions:", total_fraud)
print("Output File:", output_file)

Processed: 100,000 | Fraud: 116
Processed: 200,000 | Fraud: 147
Processed: 300,000 | Fraud: 181
Processed: 400,000 | Fraud: 206
Processed: 500,000 | Fraud: 233
Processed: 600,000 | Fraud: 361
Processed: 700,000 | Fraud: 417
Processed: 800,000 | Fraud: 460
Processed: 900,000 | Fraud: 498
Processed: 1,000,000 | Fraud: 535
Processed: 1,100,000 | Fraud: 1,488
Processed: 1,200,000 | Fraud: 1,518
Processed: 1,300,000 | Fraud: 1,548
Processed: 1,400,000 | Fraud: 1,572
Processed: 1,500,000 | Fraud: 1,608
Processed: 1,600,000 | Fraud: 1,764
Processed: 1,700,000 | Fraud: 1,812
Processed: 1,800,000 | Fraud: 1,862
Processed: 1,900,000 | Fraud: 1,890
Processed: 2,000,000 | Fraud: 2,036
Processed: 2,100,000 | Fraud: 2,063
Processed: 2,200,000 | Fraud: 2,103
Processed: 2,300,000 | Fraud: 2,134
Processed: 2,400,000 | Fraud: 2,262
Processed: 2,500,000 | Fraud: 2,278
Processed: 2,600,000 | Fraud: 2,316
Processed: 2,700,000 | Fraud: 2,351
Processed: 2,800,000 | Fraud: 2,451
Processed: 2,900,000 | Fraud: 

In [5]:
import pandas as pd

file_path = r"C:\Users\prave\OneDrive\Desktop\PAYMENT FRAUD ANALYTICS\DATA PREPARATION\OUTPUT FILE\fraud_transactions.csv"

df = pd.read_csv(file_path)

print("Fraud File Rows:", len(df))
print("Fraud Column Values:")
print(df["isFraud"].value_counts())

Fraud File Rows: 8213
Fraud Column Values:
isFraud
1    8213
Name: count, dtype: int64
